<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import sys
print(sys.executable)

c:\python314\python.exe


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [4]:
mlflow.set_experiment(
    "assignment"
)

2026/05/14 20:32:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/14 20:32:36 INFO mlflow.store.db.utils: Updating database tables
2026/05/14 20:32:40 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/rafae/OneDrive/Documentos/CESAR/Códigos/ML/atividade-03-mlp-Rafabvidal/notebooks/mlruns/1', creation_time=1778801560592, experiment_id='1', last_update_time=1778801560592, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    # Carrega o Fashion MNIST
    X, y = fetch_openml(
        "Fashion-MNIST",
        version=1,
        return_X_y=True,
        as_frame=False
    )

    # Separação treino e validação
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_val, y_train, y_val

In [6]:
X_train, X_val, y_train, y_val = load_data(seed=42)

print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(56000, 784)
(14000, 784)
(56000,)
(14000,)


É necessário normalizar os dados para esse tipo de modelo? Justifique.

Sim, é necessário normalizar os dados para modelos do tipo MLP. O Fashion MNIST possui pixels com valores entre 0 e 255, e redes neurais são sensíveis à escala dos dados. Sem normalização, o treinamento pode ficar mais lento e instável, dificultando a convergência do algoritmo de otimização. Ao normalizar os valores, por exemplo dividindo os pixels por 255 para que fiquem entre 0 e 1, o modelo aprende de forma mais eficiente e estável, além de normalmente apresentar melhor desempenho.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [7]:
from sklearn.neural_network import MLPClassifier

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=50
    )

    model.fit(X_train, y_train)

    return model

In [8]:
model = train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    seed=42
)

print(model)

MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=50, random_state=42)


# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate(model, X_test, y_test):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

In [10]:
results = evaluate(model, X_val, y_val)

print(results)

{'accuracy': 0.871, 'precision': 0.8775700446803126, 'recall': 0.871, 'f1_score': 0.8726325072600166}


A função evaluate foi implementada para avaliar o desempenho do modelo no conjunto de validação. Primeiro, o modelo realiza as predições com model.predict(X_test). Em seguida, são calculadas as métricas accuracy, precision, recall e f1-score. Como o problema possui múltiplas classes, foi utilizado average="weighted" nas métricas de precision, recall e f1-score, considerando o peso de cada classe no conjunto avaliado. O resultado final é retornado em formato de dicionário, facilitando o uso posterior no MLflow.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [11]:
import time
import mlflow

def run_experiment(
    X_train,
    X_val,
    y_train,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    max_iter,
    batch_size,
    seed
):
    with mlflow.start_run():

        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed
        )

        model.fit(X_train, y_train)

        training_time = time.time() - start_time

        metrics = evaluate(model, X_val, y_val)

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        mlflow.log_metric("accuracy", metrics["accuracy"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall", metrics["recall"])
        mlflow.log_metric("f1_score", metrics["f1_score"])
        mlflow.log_metric("training_time", training_time)

        return model, metrics

In [12]:
model, metrics = run_experiment(
    X_train,
    X_val,
    y_train,
    y_val,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    max_iter=50,
    batch_size=128,
    seed=42
)

print(metrics)

{'accuracy': 0.8759285714285714, 'precision': 0.8762556200751741, 'recall': 0.8759285714285714, 'f1_score': 0.8755264994331892}


# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [13]:
activations = ["logistic", "tanh", "relu"]

results = []

for activation in activations:
    model, metrics = run_experiment(
        X_train,
        X_val,
        y_train,
        y_val,
        activation=activation,
        hidden_layers=(128, 64),
        learning_rate=0.001,
        max_iter=50,
        batch_size=128,
        seed=42
    )

    results.append({
        "activation": activation,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"]
    })

results_df = pd.DataFrame(results)
results_df

,activation,accuracy,precision,recall,f1_score
0,logistic,0.796500,0.800036,0.796500,0.793463
1,tanh,0.758500,0.761264,0.758500,0.753165
2,relu,0.875929,0.876256,0.875929,0.875526


In [14]:
results_df.sort_values(by="f1_score", ascending=False)

,activation,accuracy,precision,recall,f1_score
2,relu,0.875929,0.876256,0.875929,0.875526
0,logistic,0.796500,0.800036,0.796500,0.793463
1,tanh,0.758500,0.761264,0.758500,0.753165


Foram comparadas as funções de ativação logistic, tanh e relu utilizando os mesmos hiperparâmetros em todos os experimentos. As métricas accuracy, precision, recall e f1-score foram registradas no MLflow para permitir comparação e rastreabilidade dos resultados. Em geral, a função relu tende a apresentar melhor desempenho e treinamento mais eficiente em modelos MLP.

## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

- A ativação relu apresentou a melhor convergência, obtendo os maiores valores de accuracy, precision, recall e f1-score.
- A ativação relu também apresentou maior estabilidade, pois teve desempenho superior e mais consistente em todas as métricas avaliadas.
- Sim. Houve diferenças significativas entre os treinamentos. A função relu teve desempenho consideravelmente melhor que logistic e tanh, enquanto tanh apresentou os piores resultados entre as três ativações testadas.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [15]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

architecture_results = []

for hidden_layers in architectures:
    model, metrics = run_experiment(
        X_train,
        X_val,
        y_train,
        y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        max_iter=50,
        batch_size=128,
        seed=42
    )

    architecture_results.append({
        "hidden_layers": hidden_layers,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"]
    })

architecture_results_df = pd.DataFrame(architecture_results)
architecture_results_df

,hidden_layers,accuracy,precision,recall,f1_score
0,"(32,)",0.818500,0.825995,0.818500,0.818770
1,"(64,)",0.850214,0.852170,0.850214,0.848964
2,"(128, 64)",0.875929,0.876256,0.875929,0.875526
3,"(256, 128)",0.881071,0.881957,0.881071,0.880750


## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

- Sim, redes maiores melhoraram os resultados neste experimento. O aumento da quantidade de neurônios e camadas levou a melhores métricas de accuracy e f1-score.
- Porém, redes maiores não garantem sempre melhores resultados. Arquiteturas muito grandes podem aumentar o tempo de treinamento e causar overfitting, dependendo do problema e dos dados.
- A arquitetura (128, 64) apresentou o melhor tradeoff, pois obteve desempenho muito próximo da (256, 128), mas com menor complexidade computacional e menor custo de treinamento.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [16]:
learning_rates = [0.1, 0.01, 0.001]

lr_results = []

for lr in learning_rates:
    model, metrics = run_experiment(
        X_train,
        X_val,
        y_train,
        y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=50,
        batch_size=128,
        seed=42
    )

    lr_results.append({
        "learning_rate": lr,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"]
    })

lr_results_df = pd.DataFrame(lr_results)
lr_results_df

,learning_rate,accuracy,precision,recall,f1_score
0,0.100,0.100000,0.010000,0.100000,0.018182
1,0.010,0.655429,0.716806,0.655429,0.607367
2,0.001,0.875929,0.876256,0.875929,0.875526


## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

- Sim. O treinamento ficou instável com learning rate 0.1, apresentando métricas muito baixas, indicando que o modelo não conseguiu aprender adequadamente.
- Sim. Houve dificuldade de convergência principalmente com 0.1, pois a taxa de aprendizado muito alta provavelmente fez o modelo oscilar durante a otimização. O valor 0.01 apresentou melhora, mas ainda inferior ao melhor resultado.
- O learning rate 0.001 apresentou o melhor comportamento, obtendo os maiores valores de accuracy, precision, recall e f1-score, além de maior estabilidade durante o treinamento.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


- A ativação relu apresentou o melhor desempenho, obtendo os maiores valores de accuracy, precision, recall e f1-score.
- A arquitetura (128, 64) apresentou o melhor tradeoff entre desempenho e custo computacional, alcançando bons resultados sem aumentar excessivamente a complexidade do modelo.
- O learning rate 0.001 apresentou maior estabilidade e melhor convergência durante o treinamento.
- Não foram observados sinais claros de overfitting, pois o modelo manteve bom desempenho no conjunto de validação.
- A melhor configuração final foi:
   ativação: relu /
   arquitetura: (128, 64) /
   learning rate: 0.001
- As principais dificuldades observadas foram a instabilidade causada por learning rates altos e a diferença de desempenho entre funções de ativação, além do maior custo computacional em arquiteturas maiores.